In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import sys
sys.path.append("../../../")

from src.data.data_splits import generate_split_mask
np.random.seed(0)
plt.rcParams["font.size"] = 7

In [ ]:
base_path = Path("../../../data/raw/mimic-ecg")
df = pd.read_csv(base_path / "record_list.csv")
mach_df = pd.read_csv(base_path / "machine_measurements.csv", low_memory=False)
patients_df = pd.read_csv("../../../data/raw/mimiciv/3.1/patients.csv")

In [ ]:
mach_df

In [ ]:
print(f"{len(df)} ECGs from {df.subject_id.nunique()} patients")

In [ ]:
# merge report columns
mach_df["report"] = mach_df[
        mach_df.columns[mach_df.columns.str.startswith('report_')]
    ].fillna('').agg('; '.join, axis=1).str.lower().replace(r'(;\s*)+', '; ', regex=True).str.strip('; ')

In [ ]:
i = np.random.randint(len(mach_df), size=500)
mach_df.loc[i, "report"].values

In [ ]:
substrings = [
    "sinus",
    "tachycardia",
    "bradycardia",
    "atrial fibrillation",
    "atrial flutter",
    "av block",
    "a-v block",
    "rbbb",
    "lbbb",
    "pacemaker",
    "infarct",
    "pvc",
    "abnormal ecg",
    "borderline ecg",
    "lateral st-t changes",
    "fascicular block",
]
for s in substrings:
    mach_df[s] = mach_df["report"].str.contains(s, na=False)
# merge column 'av block' and 'a-v block'
mach_df["av block"] = mach_df["av block"] | mach_df["a-v block"]
mach_df = mach_df.drop(columns=["a-v block"])
substrings.remove("a-v block")

In [ ]:
counts = []
for s in substrings:
    scount = mach_df[s].sum()
    counts.append(scount)
    print(f"{s}: {scount}")

In [ ]:
print(f"total of {len(substrings)} classes")
print(substrings)

In [ ]:
plt.figure(figsize=(3.5, 2))
plt.barh(substrings, counts)
plt.xlabel("Counts")
plt.xscale("log")

In [ ]:
mach_df

In [ ]:
assert df.subject_id.equals(mach_df.subject_id)

In [ ]:
for s in substrings:
    df[s] = mach_df[s]

In [ ]:
df['ecg_time'] = pd.to_datetime(df['ecg_time'])
df = pd.merge(df, patients_df, on='subject_id')

In [ ]:
df

In [ ]:
df['ecg_time']

In [ ]:
# calcuate age at acquisition using anchor age
df = df.rename(columns={'gender': 'sex'})
df = df.rename(columns={'anchor_age': 'age'})
study_year = df.ecg_time.dt.year
delta_years = study_year - df['anchor_year']
df['age'] = df['age'] + delta_years

In [ ]:
df.age.plot.hist(figsize=(3,2))

In [ ]:
df.sex.value_counts()

### Number of records per patient

In [ ]:
import scipy
plt.figure(figsize=(2.5, 1.75))
counts = df.groupby('subject_id').size().values
mean_count = np.mean(counts)
median_count = np.median(counts)
percentile = np.percentile(counts, 90)
# empirical survival function of counts
res = scipy.stats.ecdf(counts)
res.sf.plot(color="blue")
plt.axvline(mean_count, color='red', label=f"Mean: {mean_count:.2f}")
plt.axvline(median_count, color='green', label=f"Median: {median_count}")
plt.axvline(percentile, color='orange', label=f"90th perc: {percentile}")
plt.xlabel("Number of ECGs per Patient")
plt.ylabel("1 - Cumulative Probability")
plt.legend()
plt.yscale("log")
plt.title("MIMIC-ECG: #ECGs per Patient")
plt.savefig("../../../figs/mimic-ecg/mimic-ecg_record_stats.pdf", bbox_inches='tight')

### Data splits

In [ ]:
print(f"total: {len(df)} ECGs from {df.subject_id.nunique()} patients")

In [ ]:
# patient-level train/val-test split
unique_subjects = df.subject_id.unique()
np.random.shuffle(unique_subjects)
n_val, n_test = 5_000, 10_000
test_pids = unique_subjects[:n_test]
val_pids = unique_subjects[n_test : n_test + n_val]
train_pids = unique_subjects[n_test + n_val :]

traindf = df[df.subject_id.isin(train_pids)]
val_df = df[df.subject_id.isin(val_pids)]
test_df = df[df.subject_id.isin(test_pids)]

traindf.reset_index(drop=True, inplace=True)
val_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)

# temporal patient-specific split
eval_mask = generate_split_mask(
    dataframe=traindf,
    patient_id_col="subject_id",
    timestamp_col="ecg_time",
    label_cols=substrings,
    n_holdout_classes=1,
)

train_historical_df = traindf[~eval_mask]
train_future_df = traindf[eval_mask]
train_historical_df.reset_index(drop=True, inplace=True)
train_future_df.reset_index(drop=True, inplace=True)

print(f"Train(historical): {len(train_historical_df)} ECGs from {train_historical_df.subject_id.nunique()} patients")
print(f"Val: {len(val_df)} ECGs from {val_df.subject_id.nunique()} patients")
print(f"Test: {len(test_df)} ECGs from {test_df.subject_id.nunique()} patients")
print(
    f"Train(future): {len(train_future_df)} ECGs from {train_future_df.subject_id.nunique()} patients"
)

In [ ]:
# we create a helper dataframe that indicates all/any diseases present per patient in training(historical)
present_diseases_by_patient = train_historical_df.groupby('subject_id')[substrings].max()
present_diseases_by_patient

In [ ]:
from typing import List


# we generate two new columns for each future record
    # 'seen_diseases': comma-separated list of disease names that were already present in the historical records of the same patient
    # 'unseen_diseases': comma-separated list of disease names that are present in the future record but were not present in the historical records of the same patient

def case_stratification_by_historical_disease_presence(row:pd.Series, historical_diseases_by_patient:pd.DataFrame, label_cols:List[str], patient_id_col:str="subject_id", verbose:bool=False) -> bool:
    pid = row[patient_id_col]
    label = row[label_cols]
    present_diseases_historical = historical_diseases_by_patient.loc[pid]
    assert present_diseases_historical.shape == label.shape, f"Shape mismatch: {present_diseases_historical.shape} vs {label.shape}"
    positive_unseen = label[(label == 1) & (present_diseases_historical == 0)]
    positive_seen = label[((label == 1) & (present_diseases_historical == 1)) | ((label == 0) & (present_diseases_historical == 0))]
    print(f"Patient {pid} has unseen diseases: {positive_unseen.index.tolist()}") if verbose and len(positive_unseen) > 0 else None
    print(f"Patient {pid} has seen diseases: {positive_seen.index.tolist()}") if verbose and len(positive_seen) > 0 else None
    unseen_labels = positive_unseen.index.tolist() if len(positive_unseen) > 0 else []
    seen_labels = positive_seen.index.tolist() if len(positive_seen) > 0 else []
    assert set(unseen_labels).isdisjoint(set(seen_labels)), f"Seen and unseen labels should be disjoint: {unseen_labels} vs {seen_labels}"
    unseen_labels_str = ",".join(unseen_labels)
    seen_labels_str = ",".join(seen_labels)
    return pd.Series([seen_labels_str, unseen_labels_str]) # must return pd.Series to fill two columns simultaneously

In [ ]:
train_future_df[['seen_diseases', 'unseen_diseases']] = train_future_df.progress_apply(
    lambda row: case_stratification_by_historical_disease_presence(
        row,
        historical_diseases_by_patient=present_diseases_by_patient,
        label_cols=substrings,
        patient_id_col="subject_id",
        verbose=False
    ),
    axis=1,
    result_type='expand'
)
train_future_df['has_unseen_disease'] = train_future_df['unseen_diseases'].apply(lambda x: len(x) > 0)
train_future_df['has_seen_disease'] = train_future_df['seen_diseases'].apply(lambda x: len(x) > 0)

In [ ]:
train_future_df['has_unseen_disease'].value_counts()

In [ ]:
train_future_df['has_seen_disease'].value_counts()

In [ ]:
train_future_df.unseen_diseases.value_counts()

In [ ]:
train_future_df.seen_diseases.value_counts()

In [ ]:
csv_root = Path("../../../data/csv")
train_historical_df.to_csv(csv_root / "mimic-ecg_train_historical.csv", index=False)
train_future_df.to_csv(csv_root / "mimic-ecg_train_future.csv", index=False)
val_df.to_csv(csv_root / "mimic-ecg_val.csv", index=False)
test_df.to_csv(csv_root / "mimic-ecg_test.csv", index=False)